# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset schema is provided via a Croissant (JSON-LD) URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
We'll load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List all available record sets by their `@id` and show their fields by `@id`
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets:")
all_record_set_ids = []
for record_set in record_sets:
    print(f"  RecordSet @id: {record_set['@id']} | Name: {record_set.get('name', '<no name>')}")
    all_record_set_ids.append(record_set['@id'])
    # List fields within each record set, by @id
    if 'field' in record_set:
        fields = record_set['field']
        print(f"    Fields (@id):")
        for field in fields:
            if isinstance(field, dict):
                print(f"      - {field['@id']}")
            else:
                print(f"      - {field}")
    print('-' * 40)

# For demonstration, print a few example records from the first record set (if present)
if all_record_set_ids:
    example_rs_id = all_record_set_ids[0]
    print(f"\nExample records for record set @id: {example_rs_id}")
    for i, record in enumerate(dataset.records(record_set=example_rs_id)):
        print(record)
        if i >= 2:  # Show just 3 example records
            break

## 3. Data Extraction
Let's load the full data from each record set into a Pandas DataFrame. We'll use `@id` for referencing each record set.

In [ ]:
# Extract data from each record set by its @id
dataframes = {}
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id}")

# Choose a record set (by @id) for analysis; we'll use the first one for this example:
primary_record_set_id = all_record_set_ids[0] if all_record_set_ids else None

if primary_record_set_id:
    print(f"Columns for {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    print(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Common data processing steps—such as filtering numeric values, normalization, removing outliers, and grouping—rely on knowing which fields are available and their `@id`.

We'll select a numeric field and a grouping field by their `@id` values for demonstration. Update the `numeric_field_id` and `group_field_id` to actual field IDs you want to analyze, as obtained above.

In [ ]:
# Manually set the field @ids based on the DataFrame columns (inspect above outputs for actual @id values)

# Example: Pick a column that looks numeric from the DataFrame.
df = dataframes[primary_record_set_id]
numeric_columns = df.select_dtypes(include='number').columns
if len(numeric_columns) == 0:
    # Try to infer numeric columns if there was a type conversion issue:
    possible_numeric = [col for col in df.columns if df[col].apply(lambda x: pd.api.types.is_number(x) or pd.api.types.is_float(x) if pd.notna(x) else False).sum() > 0]
    numeric_columns = possible_numeric

if not numeric_columns.empty:
    numeric_field_id = numeric_columns[0]   # Use first numeric field for demonstration
else:
    numeric_field_id = None

# For grouping field, pick a likely categorical column by @id (example: ward, gender, or similar)
categorical_columns = df.select_dtypes(include='object').columns
group_field_id = None
for col in categorical_columns:
    if df[col].nunique() > 1 and df[col].nunique() < 20:
        group_field_id = col
        break

if numeric_field_id and numeric_field_id in df.columns:
    # Filter records where value is above a threshold
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered out records where {numeric_field_id} > {threshold:.2f} (using @id)")
    print(filtered_df.head())

    # Normalize this numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the chosen field (if suitable column exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id} (@id):")
        print(grouped_df.head())
else:
    print("No suitable numeric field available for EDA.")

## 5. Visualization
Let's visualize some distributions or relationships in the data, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot the distribution of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion

- This notebook demonstrated how to use the `mlcroissant` library to explore croissant-based datasets using only their Croissant schema URL.
- All data entities—record sets, fields, columns—were referenced *exclusively* by their `@id` to ensure portability and clarity.
- The FAIR² dataset enables analyses of rangeland management, and users can extend this notebook to perform modeling, deeper EDA, or policy analytics as needed.

**Next steps:**
- Consult the dataset documentation to interpret specific `@id` field meanings.
- Perform deeper domain-specific analyses using the loaded data.